<div style="text-align: center;" >
<h1 style="margin-top: 0.2em; margin-bottom: 0.1em;">Rocket Launch Data Collection</h1>


</div>
<br>

## 1.0 Import Libraries

In [13]:
import numpy as np
import pandas as pd
import requests
import time
import json
import os

## 1.1 File Setup and API Configuration

In [14]:
# Setting up save file and starting url of LL2
file_name = "launches_DEMO.json"
base_url = "https://ll.thespacedevs.com/2.2.0/launch/"

# Start and End windows for data collection
start_date = "2025-09-01T00:00:00Z"
end_date = "2025-12-31T23:59:59Z"

## 1.2 Save Function

In [16]:
# Create function to save data to json incrementally to avoid losing progress due to:
# 1. Rate limits (15 calls/hour)
# 2. Script crashes

def save_data(data_list):
    with open(file_name, "w") as f:
        json.dump(data_list, f, indent=4)

## 1.3 Load Existing Data (Resume Logic)

In [17]:
if os.path.exists(file_name):
    with open(file_name, "r") as f:
        all_launches = json.load(f)
    print(f"Resuming with {len(all_launches)} launches already saved.")
else:
    all_launches = []

## 1.4 API Parameters

In [18]:
# Parameters for collection from LL2
params = {
    "window_start__gte": start_date,
    "window_start__lte": end_date,
    "limit": 50,                      # Number of observations per request
    "ordering": "window_start",       # Sort by ascending
    "offset": len(all_launches)       # If we have already collected some data, this will tell the loop to skip all these rows.
}

## 1.5 Data Collection Loop

In [19]:

current_url = base_url

while current_url:
    print(f"Requesting: {current_url}")
    response = requests.get(current_url, params=params)

    # Handle rate limit error (429)
    if response.status_code == 429:
        print("Rate limit hit. Sleeping for 5 minutes before checking again...")
        time.sleep(300)
        continue
    
    # Handle all other errors
    if response.status_code != 200:
        print(f"Error {response.status_code}: {response.text}")
        break

    data = response.json()
    all_launches.extend(data["results"])

    # Save progress after each request
    save_data(all_launches)
    print(f"Saved! Total launches collected: {len(all_launches)}")

    # Move to next page
    current_url = data.get("next")
    params = None                    # Only need for first request

    print("Waiting 2 seconds due to 15 requests/hour limit...")
    time.sleep(2)

print(f"Extraction complete. Total launches collected: {len(all_launches)}")



Requesting: https://ll.thespacedevs.com/2.2.0/launch/
Saved! Total launches collected: 50
Waiting 2 seconds due to 15 requests/hour limit...
Requesting: https://ll.thespacedevs.com/2.2.0/launch/?limit=50&offset=50&ordering=window_start&window_start__gte=2025-09-01T00%3A00%3A00Z&window_start__lte=2025-12-31T23%3A59%3A59Z
Saved! Total launches collected: 100
Waiting 2 seconds due to 15 requests/hour limit...
Requesting: https://ll.thespacedevs.com/2.2.0/launch/?limit=50&offset=100&ordering=window_start&window_start__gte=2025-09-01T00%3A00%3A00Z&window_start__lte=2025-12-31T23%3A59%3A59Z
Saved! Total launches collected: 133
Waiting 2 seconds due to 15 requests/hour limit...
Extraction complete. Total launches collected: 133


## 1.6 Normalize JSON to Dataframe

In [20]:
df = pd.json_normalize(all_launches)

## 1.7 Data Type Conversions

In [21]:
print(df.dtypes.to_string())

id                                     object
url                                    object
slug                                   object
name                                   object
last_updated                           object
net                                    object
window_end                             object
window_start                           object
probability                           float64
weather_concerns                       object
holdreason                             object
failreason                             object
hashtag                                object
webcast_live                             bool
image                                  object
infographic                            object
program                                object
orbital_launch_attempt_count          float64
location_launch_attempt_count           int64
pad_launch_attempt_count                int64
agency_launch_attempt_count             int64
orbital_launch_attempt_count_year 

In [22]:
# Change these to numeric/datetime as we need these columns to collect weather data later.
df["pad.latitude"] =  pd.to_numeric(df["pad.latitude"], errors='coerce')
df["pad.longitude"] =  pd.to_numeric(df["pad.longitude"], errors='coerce')

df["window_start"] =  pd.to_datetime(df["window_start"], errors='coerce')
df["window_end"] =  pd.to_datetime(df["window_end"], errors='coerce')

df["window_start_date"] = df["window_start"].dt.strftime("%Y-%m-%d")
df["window_end_date"] = df["window_end"].dt.strftime("%Y-%m-%d")

df.head()

,id,url,slug,name,last_updated,net,window_end,window_start,probability,weather_concerns,...,pad.location.map_image,pad.location.timezone_name,pad.location.total_launch_count,pad.location.total_landing_count,pad.country_code,pad.map_image,pad.total_launch_count,pad.orbital_launch_attempt_count,window_start_date,window_end_date
0,d7caa65c-4272-4726-b068-4f75657b896d,https://ll.thespacedevs.com/2.2.0/launch/d7caa...,shavit-2-ofek-19,Shavit-2 | Ofek-19,2025-09-03T05:05:43Z,2025-09-02T19:30:00Z,2025-09-02 19:30:00+00:00,2025-09-02 19:30:00+00:00,NaN,None,...,https://thespacedevs-prod.nyc3.digitaloceanspa...,Asia/Jerusalem,13,0,ISR,https://thespacedevs-prod.nyc3.digitaloceanspa...,13,13,2025-09-02,2025-09-02
1,3628175d-948d-4eaa-80d2-10532869d5d0,https://ll.thespacedevs.com/2.2.0/launch/36281...,falcon-9-block-5-starlink-group-17-8,Falcon 9 Block 5 | Starlink Group 17-8,2025-09-03T05:08:22Z,2025-09-03T03:51:26Z,2025-09-03 06:33:00+00:00,2025-09-03 02:33:00+00:00,NaN,None,...,https://thespacedevs-prod.nyc3.digitaloceanspa...,America/Los_Angeles,869,33,USA,https://thespacedevs-prod.nyc3.digitaloceanspa...,254,254,2025-09-03,2025-09-03
2,2a799baa-899d-4a71-8701-d15df5d3d2c4,https://ll.thespacedevs.com/2.2.0/launch/2a799...,falcon-9-block-5-starlink-group-10-22,Falcon 9 Block 5 | Starlink Group 10-22,2025-09-03T14:36:31Z,2025-09-03T11:56:10Z,2025-09-03 15:06:00+00:00,2025-09-03 11:06:00+00:00,70.0,Cumulus Cloud Rule,...,https://thespacedevs-prod.nyc3.digitaloceanspa...,America/New_York,1095,71,USA,https://thespacedevs-prod.nyc3.digitaloceanspa...,372,372,2025-09-03,2025-09-03
3,50448abf-884a-4e04-87da-3950ab7c43e7,https://ll.thespacedevs.com/2.2.0/launch/50448...,long-march-3cyz-1-shiyan-29,Long March 3C/YZ-1 | Shiyan 29,2025-11-16T15:39:57Z,2025-09-05T02:34:00Z,2025-09-05 02:53:00+00:00,2025-09-05 02:26:00+00:00,NaN,None,...,https://thespacedevs-prod.nyc3.digitaloceanspa...,Asia/Shanghai,238,0,CHN,https://thespacedevs-prod.nyc3.digitaloceanspa...,124,124,2025-09-05,2025-09-05
4,b1451c26-c4af-45ee-879e-6d1f6f8eec52,https://ll.thespacedevs.com/2.2.0/launch/b1451...,ceres-1-3-satellites-2,Ceres-1 | 3 satellites,2025-09-05T23:59:58Z,2025-09-05T11:39:00Z,2025-09-05 12:08:00+00:00,2025-09-05 11:28:00+00:00,NaN,None,...,https://thespacedevs-prod.nyc3.digitaloceanspa...,Asia/Shanghai,278,2,CHN,https://thespacedevs-prod.nyc3.digitaloceanspa...,66,66,2025-09-05,2025-09-05


## 1.8 Data Check

In [ ]:
# Check data range
print(df["window_start"].min())
print(df["window_start"].max())

# Check class distribution
print(df["status.abbrev"].value_counts())

# We have 1839 success and only 105 failures/partial failures, making our classes extremely imbalanced.

2025-09-02 19:30:00+00:00
2025-12-30 22:32:00+00:00
status.abbrev
Success    130
Failure      3
Name: count, dtype: int64


## 1.9 Save Data to CSV

In [24]:
df.to_csv("launches_DEMO.csv", index=False)